# Bases de données vectorielles : ta première démo avec Pinecone

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/cercle-des-experts/blob/main/llm-fine-tuning/bases-de-donnees-vectorielles-partie-2/bases-de-donnees-vectorielles-partie-2.ipynb)

📖 **Le mini-cours complet** : [Bases de données vectorielles : index avancés et pratique (Partie 2)](https://www.monshotdata.com/p/bases-de-donnees-vectorielles-partie-2)

🧭 **Le cours** : [LLM & fine-tuning › 1.1 Fondations des représentations de texte](https://www.monshotdata.com/mini-cours/llm-fine-tuning#h2sc11)

---

Comment un moteur de recherche comprend-il que « Quel véhicule fonctionne sans essence ? » parle d'une voiture électrique, alors qu'aucun mot n'est commun avec la phrase « La voiture électrique se recharge en vingt minutes » ? Grâce aux **embeddings** et à une **base de données vectorielle**. En une vingtaine de lignes, tu vas encoder des phrases avec un modèle récent, les ranger dans un index Pinecone et les interroger **par le sens**.

**À la fin de ce notebook, tu sauras :**

- transformer des phrases en vecteurs avec un modèle d'embedding libre (BGE-M3) ;
- créer un index vectoriel géré, y envoyer des documents et les retrouver par similarité cosinus ;
- reconnaître les bons réflexes en pratique : clé API hors du code, même modèle pour les documents et la requête, index supprimé à la fin.


### Avant de lancer

**Avant de lancer :**

- Un compte Pinecone gratuit est nécessaire, avec une clé API (menu « API Keys » du tableau de bord).
- Dans Colab, ajoute la clé dans les **secrets** (icône de clé, à gauche) sous le nom `PINECONE_API_KEY` et active l'accès au notebook. Hors Colab, place-la dans la variable d'environnement `PINECONE_API_KEY`. Ne l'écris jamais dans le notebook.
- Le modèle d'embedding se télécharge au premier lancement (environ 2,3 Go) : un GPU n'est pas nécessaire pour cette démo.
- L'index est **supprimé à la fin du notebook**, pour ne rien laisser sur ton compte.

Exécute les cellules de haut en bas.

🧪 *Testé le 20 novembre 2026 sur Colab (CPU) avec le SDK Pinecone 10.3 et BGE-M3.*

## 1. Installer les bibliothèques

Deux bibliothèques : le SDK officiel de Pinecone (version 10 ou plus) et `sentence-transformers`, qui sert à charger le modèle d'embedding.

In [1]:
# SDK Pinecone (version 10 ou plus) et sentence-transformers
!pip install -U pinecone sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 43.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 28.8 MB/s  0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sentence-transformers]


## 2. Charger le modèle d'embedding

Un **embedding** transforme un texte en une liste de nombres (un vecteur) : deux textes qui ont un sens proche donnent deux vecteurs proches. On utilise ici `BGE-M3`, un modèle libre (licence MIT) qui comprend très bien le français. Sa **dimension** (la longueur des vecteurs) est affichée en sortie : on en aura besoin pour créer l'index.

In [2]:
import os
import time

from pinecone import DenseVectorQuery, Pinecone, SchemaBuilder
from sentence_transformers import SentenceTransformer

# Modèle d'embedding libre (licence MIT)
modele = SentenceTransformer("BAAI/bge-m3")

dimension = modele.get_embedding_dimension()
print(dimension)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

1024


## 3. Lire la clé API et créer l'index

### 3.1 La clé API, sans jamais l'écrire dans le notebook

La cellule cherche d'abord un secret Colab, puis une variable d'environnement. Si aucune des deux n'existe, elle s'arrête avec un message qui explique quoi faire.

In [3]:
# Lecture de la clé API : secret Colab en premier, variable d'environnement en repli
cle_api = None
try:
    from google.colab import userdata
    cle_api = userdata.get("PINECONE_API_KEY")
except Exception:
    cle_api = None

if not cle_api:
    cle_api = os.environ.get("PINECONE_API_KEY")

if not cle_api:
    raise RuntimeError(
        "Clé API Pinecone introuvable. Dans Colab, ajoute un secret nommé "
        "PINECONE_API_KEY (icône de clé à gauche) et active l'accès au notebook. "
        "Hors Colab, définis la variable d'environnement PINECONE_API_KEY."
    )

os.environ["PINECONE_API_KEY"] = cle_api

### 3.2 L'index

Un **index** est la « table » de la base vectorielle. On décrit son schéma, qui ne déclare que les champs servant à la **recherche** : ici un seul champ vectoriel `emb` (de la dimension du modèle, comparé par **similarité cosinus**). Le texte de chaque phrase (`texte`) n'a pas à y figurer : on le joindra simplement à chaque document à l'envoi, et Pinecone l'indexera automatiquement pour qu'on puisse le récupérer dans les résultats. Le plan gratuit impose AWS, région `us-east-1`. La cellule ne crée l'index que s'il n'existe pas déjà, tu peux donc la relancer sans risque.

In [4]:
# La clé API est lue dans la variable d'environnement PINECONE_API_KEY
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

nom_index = "demo-vecteurs"

# Le schéma ne déclare que les champs servant à la recherche (ici le vecteur) ;
# le champ « texte » sera simplement joint à chaque document, sans être déclaré
schema = (
    SchemaBuilder()
    .add_dense_vector_field("emb", dimension=dimension, metric="cosine")
    .build()
)

# Plan gratuit : AWS, région us-east-1 uniquement
if not pc.indexes.exists(nom_index):
    pc.indexes.create(
        name=nom_index,
        schema=schema,
        deployment={
            "deployment_type": "managed",
            "cloud": "aws",
            "region": "us-east-1",
        },
    )

## 4. Encoder les phrases d'exemple

Huit phrases courtes, volontairement regroupées par thème : des animaux (chat, chien, lion), des véhicules (voiture, camion, moto) et des fruits (pomme, banane). Aucune ne contient le mot « animal » ni « véhicule » : c'est le sens qui devra faire le travail.

In [5]:
phrases = {
    "a": "Le chat dort sur le canapé.",
    "b": "Mon chien adore courir dans le jardin.",
    "c": "Le lion est le roi de la savane.",
    "d": "La voiture électrique se recharge en vingt minutes.",
    "e": "Ce camion transporte des fruits jusqu'au marché.",
    "f": "J'ai croqué une pomme bien mûre.",
    "g": "La banane est riche en potassium.",
    "h": "Le moteur de la moto fait beaucoup de bruit.",
}

# Un vecteur par phrase
vecteurs = modele.encode(list(phrases.values()), normalize_embeddings=True)
print(vecteurs.shape)

(8, 1024)


## 5. Envoyer les documents dans l'index (upsert)

Chaque document porte un identifiant, son vecteur et son texte. L'*upsert* insère le document, ou le met à jour s'il existe déjà. On range le tout dans un **namespace** (`demo`), un sous-espace de l'index qui permet de séparer des jeux de données. L'indexation est asynchrone : on attend quelques secondes avant d'interroger.

In [6]:
index = pc.index(nom_index)

documents = [
    {"_id": id_doc, "emb": vecteur.tolist(), "texte": texte}
    for (id_doc, texte), vecteur in zip(phrases.items(), vecteurs)
]

index.documents.upsert(namespace="demo", documents=documents)

# L'indexation est asynchrone : on laisse quelques secondes à Pinecone
time.sleep(10)

## 6. Interroger l'index

On encode la question avec **le même modèle** que les documents (c'est indispensable : sinon les vecteurs ne vivent pas dans le même espace). Puis on demande les 3 documents les plus proches.

In [7]:
requete = "Quel véhicule fonctionne sans essence ?"

# Le même modèle encode la requête
v = modele.encode([requete], normalize_embeddings=True)[0]

resultats = index.documents.search(
    namespace="demo",
    top_k=3,
    score_by=[DenseVectorQuery(field="emb", values=v.tolist())],
    include_fields=["texte"],
)

## 7. Afficher les résultats

Chaque résultat porte un score de similarité (plus il est proche de 1, plus le sens est proche). La phrase sur la voiture électrique doit arriver en tête, avec une avance nette sur les autres, même sans mot en commun avec la question. Les scores absolus dépendent du modèle : ce qui compte, c'est le classement.

In [8]:
for doc in resultats.matches:
    print(f"{doc.score:.3f}  {doc.texte}")

0.497  La voiture électrique se recharge en vingt minutes.
0.392  Ce camion transporte des fruits jusqu'au marché.
0.370  Le moteur de la moto fait beaucoup de bruit.


## 8. Supprimer l'index

L'index n'est utile que pour la démo : on le supprime pour ne rien laisser sur ton compte.

In [9]:
# On supprime l'index pour ne rien laisser traîner sur le compte
pc.indexes.delete(nom_index)

## 9. À toi de jouer

Change la phrase de la variable `requete` (par exemple « Quel fruit est riche en potassium ? » ou « Comment recharger un véhicule électrique ? ») et relance les cellules d'interrogation avant de supprimer l'index. Tu peux aussi ajouter tes propres phrases dans le dictionnaire `phrases`.

---

## Pour aller plus loin

- [Bases de données vectorielles : des embeddings aux index (Partie 1)](https://www.monshotdata.com/p/bases-de-donnees-vectorielles-partie-1) — pourquoi le kNN exact ne passe pas à l'échelle, et comment les index approximatifs y remédient
- [Bases de données vectorielles : index avancés et pratique (Partie 2)](https://www.monshotdata.com/p/bases-de-donnees-vectorielles-partie-2) — Product Quantization, HNSW, RAG et panorama du marché, le mini-cours de ce notebook